# Read and analyze New Glenn explosion data

This notebook is adapted from `25_make_plots.ipynb`, but restricted to the USF Centaur stations:

- **Trillium seismic channels**: `DHZ`, `DHN`, `DHE`, `DH1`, `DH2`
- **infraBSU / Chaparral infrasound channels**: `DD0`, `DD1`, `DD2`, `DDF`

The workflow is:

1. read and trim the high-rate MiniSEED files;
2. remove flat/dead channels;
3. apply instrument response correction;
4. measure peak amplitudes and peak/arrival times;
5. merge amplitude tables with station/source geometry;
6. map and plot spatial amplitude variations;
7. fit simple arrival-time and amplitude-decay models;
8. compare observed and modeled arrivals/amplitudes at each station.


## 1. Settings

In [1]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import obspy
from obspy import UTCDateTime, Stream

warnings.filterwarnings("ignore", category=UserWarning)

# ---------------------------------------------------------------------
# Data paths
# ---------------------------------------------------------------------
DATADIR = Path("~/mybox/KSC_USF_UCF/Day 149 Data").expanduser()
OUTDIR = DATADIR / "newglenn_plots"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Raw high-rate Centaur day files. These are the USF stations only.
RAW_GLOB = "1R.B*.00.D*.D.2026.149"
TRIMMED_PREFIX = "trimmed_"

# ---------------------------------------------------------------------
# Event timing
# ---------------------------------------------------------------------
# Adjust EXPLOSION_TIME if a better origin time is available.
# The old notebook used 00:59--01:03 UTC and then zoomed around 01:00:30--01:01:30.
EXPLOSION_TIME = UTCDateTime(2026, 5, 29, 1, 0, 30)

# Broader read/trim window
PRE_EVENT_S = 120
POST_EVENT_S = 180
STARTTRIM = EXPLOSION_TIME - PRE_EVENT_S
ENDTRIM = EXPLOSION_TIME + POST_EVENT_S

# Search windows for peak measurements relative to EXPLOSION_TIME.
# These deliberately overlap enough to tolerate imperfect origin time.
SEISMIC_SEARCH = (-10, 90)       # direct seismic / early ground motion
INFRASOUND_SEARCH = (20, 140)    # acoustic / ground-coupled airwave

# Sliding window used to measure "event amplitude" rather than one-sample spikes.
WINDOW_SEC = 1.0

# ---------------------------------------------------------------------
# Geometry
# ---------------------------------------------------------------------
# Preferred: read station and source geometry from an Excel file, if available.
# Set these to match your local metadata workbook.
KSC_DIR = Path("~/Dropbox/KSC_paper").expanduser()
GEOMETRY_XLS = KSC_DIR / "KSC_seismic_network_2016_2022.xls"
STATION_SHEET = "newglenn"       # change to "artemis2" or your current USF deployment sheet
SOURCE_SHEET = "launchpads_2026"
SOURCE_NAME = "LC36"             # New Glenn pad; change if your sheet uses a different label

# Fallback source coordinates. Replace with the exact explosion/source coordinates if the Excel read fails.
# These are deliberately set to NaN so the notebook fails loudly rather than silently using bad geometry.
SOURCE_LAT = np.nan
SOURCE_LON = np.nan

# ---------------------------------------------------------------------
# Plotting controls
# ---------------------------------------------------------------------
MAKE_RAW_STATION_PLOTS = False
THRESHOLD_FLAT = 10.0
LABEL_POINTS = True

print(f"DATADIR = {DATADIR}")
print(f"OUTDIR  = {OUTDIR}")

DATADIR = /Users/glennthompson/mybox/KSC_USF_UCF/Day 149 Data
OUTDIR  = /Users/glennthompson/mybox/KSC_USF_UCF/Day 149 Data/newglenn_plots


## 2. Read, trim, and cache the New Glenn MiniSEED files

In [2]:
def read_trim_and_cache(datadir: Path, raw_glob: str, starttime, endtime, *, overwrite=False) -> Stream:
    """
    Read raw day files, trim to the event window, and write trimmed MiniSEED files.

    Returns a merged Stream containing the trimmed traces.
    """
    st_all = Stream()
    raw_files = sorted(datadir.glob(raw_glob))
    if not raw_files:
        raise FileNotFoundError(f"No raw files matched {datadir / raw_glob}")

    for file in raw_files:
        trimmed_file = datadir / f"{TRIMMED_PREFIX}{file.name}"
        if trimmed_file.exists() and not overwrite:
            print(f"Using cached {trimmed_file.name}")
            st_all += obspy.read(str(trimmed_file))
            continue

        try:
            print(f"Reading {file.name}")
            st = obspy.read(str(file))
            st.trim(starttime=starttime, endtime=endtime)
            st.detrend("demean")
            st.write(str(trimmed_file), format="MSEED")
            st_all += st
        except Exception as e:
            print(f"Error reading {file.name}: {e}")

    st_all.merge(fill_value="interpolate")
    st_all.sort()
    return st_all


stmaster = read_trim_and_cache(DATADIR, RAW_GLOB, STARTTRIM, ENDTRIM, overwrite=False)
print(stmaster)

if MAKE_RAW_STATION_PLOTS:
    stmaster.plot(type="relative", starttime=STARTTRIM, endtime=ENDTRIM, equal_scale=False);

FileNotFoundError: No raw files matched /Users/glennthompson/mybox/KSC_USF_UCF/Day 149 Data/1R.B*.00.D*.D.2026.149

## 3. Remove flat/dead channels

In [ ]:
def get_stations(stream: Stream):
    return sorted(set(tr.stats.station for tr in stream))


def is_flat_trace(trace_in, starttime=None, endtime=None, threshold=3.0):
    """
    Identify mostly-flat channels using peak/RMS after light conditioning.

    A low peak/RMS ratio means the channel has little impulsive content in the
    event window and is often an unused/dead input.
    """
    tr = trace_in.copy()
    tr.detrend(type="linear")
    tr.taper(0.02)

    fs = tr.stats.sampling_rate
    if fs > 2:
        tr.filter("highpass", freq=0.5, corners=2, zerophase=True)

    if starttime is not None and endtime is not None:
        tr.trim(starttime, endtime)

    data = tr.data.astype(float)
    data = data[np.isfinite(data)]

    if len(data) == 0:
        return True

    rms = np.sqrt(np.mean(data**2))
    peak = np.max(np.abs(data))

    if rms == 0:
        return True

    return (peak / rms) < threshold


st_event = Stream()
for station in get_stations(stmaster):
    st_station = stmaster.select(station=station)
    for tr in st_station:
        if is_flat_trace(tr, STARTTRIM, ENDTRIM, threshold=THRESHOLD_FLAT):
            print(f"Skipping flat trace {tr.id}  (peak/RMS < {THRESHOLD_FLAT})")
        else:
            st_event.append(tr.copy())

st_event.sort()
print(st_event)

## 4. Instrument correction for USF infrasound and Trillium seismic channels

### Infrasound response correction and B20 input-range history

The infraBSU correction below retains the original 1--2 Hz pre/post response-removal scaling because the NRL infraBSU response has the correct frequency dependence but an unreliable overall sensitivity.

For a 0.5-inch infraBSU on a Nanometrics Centaur, the working overall sensitivities are **736 counts/Pa at 1 V p-p** and **18.4 counts/Pa at 40 V p-p**. B20 has no DD0 in this deployment: **DDF is the Chaparral M25 and DD1/DD2 are infraBSUs**.

B20 Sensor B was directly observed on **2026-09-23** to still be configured at **1 V p-p**. It was changed during that field visit to **40 V p-p** so that the Chaparral could retain its full dynamic range. Thus the May 28/29 New Glenn event should use **1 V p-p** for all B20 infrasound channels. The precise clock time of the September 23 change has not yet been entered, so traces recorded on that date are treated as ambiguous unless an override is supplied.

For the Chaparral M25, the NRL response is trusted directly; the empirical infraBSU rescaling is **not** applied to DDF.

In [ ]:
from flovopy.stationmetadata.build import NRL2inventory


# Empirical overall sensitivities for 0.5-inch infraBSU + Centaur.
# Increasing the Centaur input range from 1 to 40 V p-p reduces counts/Pa by 40.
INFRABSU_SENSITIVITY_COUNTS_PER_PA = {
    1: 736.0,
    40: 18.4,
}

# B20 Sensor B was observed at 1 V p-p on 2026-09-23 and changed to
# 40 V p-p during that field visit. The exact clock time is not yet recorded.
B20_VPP_CHANGE_DATE = UTCDateTime(2026, 9, 23)
B20_VPP_CHANGE_DATE_END = UTCDateTime(2026, 9, 24)


def get_b20_sensor_b_vpp(tr, override=None):
    """Return the Centaur Sensor B input range for a B20 infrasound trace.

    B20 Sensor B contains DDF (Chaparral M25) and DD1/DD2 (infraBSU).

    Known history:
      * before 2026-09-23: 1 V p-p
      * after the 2026-09-23 field change: 40 V p-p

    Because the exact change time on 2026-09-23 has not yet been entered,
    traces touching that UTC date require an explicit override.
    """
    if override is not None:
        if override not in INFRABSU_SENSITIVITY_COUNTS_PER_PA:
            raise ValueError("B20 Vpp override must be 1 or 40")
        return override

    if tr.stats.endtime < B20_VPP_CHANGE_DATE:
        return 1

    if tr.stats.starttime >= B20_VPP_CHANGE_DATE_END:
        return 40

    raise ValueError(
        f"{tr.id} overlaps 2026-09-23, when B20 Sensor B was changed "
        "from 1 to 40 V p-p. Supply b20_vpp_override=1 or 40 once the "
        "field-change time is known."
    )


def correct_infrabsu_stream(st_event: Stream, b20_vpp_override=None) -> Stream:
    """Correct USF infrasound channels to pressure (Pa).

    Sensor handling
    ---------------
    B20 DDF
        Chaparral M25. Use the NRL response directly, with the historical
        Centaur Sensor B input range. Do NOT apply the empirical infraBSU
        overall-sensitivity correction.

    B20 DD1/DD2
        0.5-inch infraBSUs. They share the same Sensor B input-range setting
        as the Chaparral: 1 V p-p before the 2026-09-23 field change and
        40 V p-p afterward.

    Other DD0/DD1/DD2 channels
        0.5-inch infraBSUs, assumed here to use 1 V p-p unless station
        metadata says otherwise.

    infraBSU correction
    -------------------
    The NRL response is retained for frequency dependence. After
    remove_response(), a 1--2 Hz pre/post amplitude comparison is used to
    adjust the overall gain to the empirical sensitivity appropriate for
    the selected Centaur range (736 counts/Pa at 1 V p-p, 18.4 counts/Pa
    at 40 V p-p).

    If the infraBSU NRL response cannot be constructed/applied, fall back to
    direct counts-to-Pa scaling using the same empirical overall sensitivity.
    """
    st_corr = st_event.select(
        network="1R",
        station="B*",
        channel="DD[012F]",
    ).copy()

    for tr in st_corr:
        if tr.stats.get("corrected", False):
            continue

        sta = tr.stats.station
        chan = tr.stats.channel

        # --------------------------------------------------------------
        # Sensor type and Centaur input range
        # --------------------------------------------------------------
        if sta == "B20":
            vpp = get_b20_sensor_b_vpp(tr, override=b20_vpp_override)

            if chan == "DDF":
                sensor = "Chaparral"
                is_infrabsu = False
                expected_sensitivity = None
            else:
                # B20 has DD1/DD2 infraBSUs in this deployment; no DD0.
                sensor = "infrabsu"
                is_infrabsu = True
                expected_sensitivity = INFRABSU_SENSITIVITY_COUNTS_PER_PA[vpp]
        else:
            sensor = "infrabsu"
            is_infrabsu = True
            vpp = 1
            expected_sensitivity = INFRABSU_SENSITIVITY_COUNTS_PER_PA[vpp]

        msg = f"Correcting {tr.id}: sensor={sensor}, Vpp={vpp}"
        if expected_sensitivity is not None:
            msg += f", target={expected_sensitivity:g} counts/Pa"
        print(msg)

        fs = tr.stats.sampling_rate
        nyq = 0.5 * fs
        pre_filt = (0.01, 0.02, 0.94 * nyq, 0.98 * nyq)

        # For infraBSU only, measure raw-count amplitude in a stable 1--2 Hz
        # band before response removal. This is used to repair the NRL
        # overall sensitivity while preserving its frequency dependence.
        before = None
        if is_infrabsu:
            tr_before = tr.copy()
            tr_before.detrend("linear")
            tr_before.taper(0.05)
            tr_before.filter("bandpass", freqmin=0.005, freqmax=0.94 * nyq)
            tr_before.filter("bandpass", freqmin=1.0, freqmax=2.0)
            tr_before.trim(starttime=starttrim, endtime=endtrim)
            before = np.nanstd(tr_before.data)

        # --------------------------------------------------------------
        # Build and apply the NRL response
        # --------------------------------------------------------------
        try:
            inv = NRL2inventory(
                nrl_path=None,
                net=tr.stats.network,
                sta=tr.stats.station,
                loc=tr.stats.location,
                chans=[tr.stats.channel],
                datalogger="Centaur",
                sensor=sensor,
                Vpp=vpp,
                fsamp=tr.stats.sampling_rate,
                lat=0.0,
                lon=0.0,
                elev=0.0,
                depth=0.0,
                sitename="",
                ondate=tr.stats.starttime,
                offdate=tr.stats.endtime,
                sensitivity=None,
                units="Pa",
            )

            tr.remove_response(
                inventory=inv,
                output="DEF",
                pre_filt=pre_filt,
                zero_mean=True,
                taper=True,
            )
            response_removed = True

        except Exception as e:
            response_removed = False
            print(f"Could not remove response for {tr.id}: {e}")

        # --------------------------------------------------------------
        # Chaparral: trust NRL response as-is
        # --------------------------------------------------------------
        if not is_infrabsu:
            if not response_removed:
                print(
                    f"No fallback sensitivity is defined for {tr.id} "
                    "(Chaparral); leaving this trace uncorrected."
                )
                continue

            tr.stats["corrected"] = True
            tr.stats["units"] = "Pa"
            tr.stats["sensor_type"] = "Chaparral M25"
            tr.stats["centaur_vpp"] = vpp
            continue

        # --------------------------------------------------------------
        # infraBSU: preserve NRL frequency response, repair overall gain
        # --------------------------------------------------------------
        if response_removed:
            tr_after = tr.copy()
            tr_after.filter("bandpass", freqmin=1.0, freqmax=2.0)
            tr_after.trim(starttime=starttrim, endtime=endtrim)
            after = np.nanstd(tr_after.data)

            if (
                np.isfinite(before)
                and np.isfinite(after)
                and before > 0
                and after > 0
            ):
                nrl_effective_counts_per_pa = before / after
                additional_factor = (
                    nrl_effective_counts_per_pa / expected_sensitivity
                )

                print(
                    f"  NRL effective sensitivity ~{nrl_effective_counts_per_pa:.3g} "
                    f"counts/Pa; target={expected_sensitivity:g}; "
                    f"additional scale={additional_factor:.6g}"
                )

                tr.data *= additional_factor
                tr.stats["empirical_sensitivity_correction"] = additional_factor
                tr.stats["nrl_effective_counts_per_pa"] = nrl_effective_counts_per_pa
            else:
                print(
                    f"Could not compute empirical infraBSU correction for {tr.id}: "
                    f"before={before}, after={after}. Leaving NRL-corrected data "
                    "without the empirical gain repair."
                )
        else:
            # Fallback: lose the frequency-dependent correction, but retain
            # the correct empirical overall counts/Pa conversion.
            print(
                f"Falling back to direct scaling for {tr.id}: "
                f"counts / {expected_sensitivity:g} counts/Pa. "
                "No frequency-dependent response correction is applied."
            )
            tr.data = tr.data.astype(float) / expected_sensitivity
            tr.stats["response_fallback"] = "direct_counts_per_pa"

        tr.stats["corrected"] = True
        tr.stats["units"] = "Pa"
        tr.stats["sensor_type"] = "infraBSU 0.5 inch"
        tr.stats["centaur_vpp"] = vpp
        tr.stats["expected_sensitivity_counts_per_pa"] = expected_sensitivity

    return st_corr


# For the 2026-05-28/29 New Glenn event, B20 is unambiguously before the
# September field change, so the function automatically uses 1 V p-p there.
st_infrabsu_corrected = correct_infrabsu_stream(st_event)
plot_stream_relative(
    st_infrabsu_corrected,
    "newglenn_infrabsu_stream_corrected.png",
    equal_scale=False,
)

## 5. Bandpass-filtered plotting streams

In [ ]:
def filtered_copy(stream: Stream, freqmin=None, freqmax=None) -> Stream:
    st = stream.copy()
    for tr in st:
        tr.detrend("linear")
        tr.taper(0.02)
        fs = tr.stats.sampling_rate
        nyq = 0.5 * fs
        if freqmin is not None and freqmax is not None:
            tr.filter("bandpass", freqmin=freqmin, freqmax=min(freqmax, 0.90 * nyq),
                      corners=4, zerophase=True)
        elif freqmin is not None:
            tr.filter("highpass", freq=freqmin, corners=4, zerophase=True)
        elif freqmax is not None:
            tr.filter("lowpass", freq=min(freqmax, 0.90 * nyq), corners=4, zerophase=True)
    return st


# Use broad bands initially; adjust after inspecting spectra.
st_infra_bp = filtered_copy(st_infra, 0.5, 20.0)
st_seis_bp = filtered_copy(st_seis, 0.5, 80.0)

st_infra_bp.plot(type="relative", starttime=EXPLOSION_TIME-20, endtime=EXPLOSION_TIME+140,
                 equal_scale=False, outfile=str(OUTDIR / "newglenn_infrasound_0p5_20Hz.png"));
st_seis_bp.plot(type="relative", starttime=EXPLOSION_TIME-20, endtime=EXPLOSION_TIME+100,
                equal_scale=False, outfile=str(OUTDIR / "newglenn_seismic_0p5_80Hz.png"));

## 6. Measure peak amplitudes and peak-window times

In [ ]:
def trace_rms(data):
    data = np.asarray(data, dtype=float)
    data = data[np.isfinite(data)]
    if len(data) == 0:
        return np.nan
    return float(np.sqrt(np.mean(data**2)))


def sliding_window_peak_index(abs_data: np.ndarray, nwin: int) -> int:
    """
    Return the start index of the contiguous window with the largest sum
    of absolute amplitudes.
    """
    abs_data = np.asarray(abs_data, dtype=float)
    nwin = int(max(1, min(nwin, len(abs_data))))
    csum = np.concatenate([[0.0], np.cumsum(abs_data)])
    window_sums = csum[nwin:] - csum[:-nwin]
    return int(np.argmax(window_sums))


def summarize_trace_amplitudes(tr, origin_time, search_window, window_sec=1.0) -> dict:
    """
    Summarize one trace in a fixed search window.

    The reported win_start/win_mid are the times of the highest-amplitude
    sliding window, which is often more robust than the single-sample peak time.
    """
    t0 = origin_time + search_window[0]
    t1 = origin_time + search_window[1]

    trw = tr.copy()
    trw.trim(starttime=t0, endtime=t1)

    data = trw.data.astype(float)
    data = np.nan_to_num(data, nan=0.0, posinf=0.0, neginf=0.0)

    if len(data) == 0:
        return {
            "SEED_id": tr.id, "network": tr.stats.network, "station": tr.stats.station,
            "location": tr.stats.location, "channel": tr.stats.channel,
            "sampling_rate": tr.stats.sampling_rate,
            "trace_start": str(tr.stats.starttime), "trace_end": str(tr.stats.endtime),
            "window_sec": window_sec, "search_start": str(t0), "search_end": str(t1),
            "pos_max": np.nan, "neg_max": np.nan, "abs_peak": np.nan,
            "median_abs": np.nan, "mean_abs": np.nan, "rms": np.nan, "std": np.nan,
            "win_start": None, "win_mid": None, "win_end": None,
            "time_since_origin_s": np.nan,
        }

    sr = trw.stats.sampling_rate
    nwin = max(1, int(round(window_sec * sr)))
    abs_data = np.abs(data)
    i0 = sliding_window_peak_index(abs_data, nwin)
    i1 = min(i0 + nwin, len(data))

    win_data = data[i0:i1]
    win_start = trw.stats.starttime + i0 / sr
    win_end = trw.stats.starttime + i1 / sr
    win_mid = win_start + 0.5 * (win_end - win_start)

    pos_max = float(np.max(data))
    neg_max = float(np.min(data))
    abs_peak = float(np.max(abs_data))

    return {
        "SEED_id": tr.id,
        "network": tr.stats.network,
        "station": tr.stats.station,
        "location": tr.stats.location,
        "channel": tr.stats.channel,
        "sampling_rate": tr.stats.sampling_rate,
        "trace_start": str(tr.stats.starttime),
        "trace_end": str(tr.stats.endtime),
        "window_sec": window_sec,
        "search_start": str(t0),
        "search_end": str(t1),
        "pos_max": pos_max,
        "neg_max": neg_max,
        "abs_peak": abs_peak,
        "median_abs": float(np.median(abs_data)),
        "mean_abs": float(np.mean(abs_data)),
        "rms": trace_rms(data),
        "std": float(np.std(data)),
        "win_rms": trace_rms(win_data),
        "win_mean_abs": float(np.mean(np.abs(win_data))),
        "win_start": str(win_start),
        "win_mid": str(win_mid),
        "win_end": str(win_end),
        "time_since_origin_s": float(win_mid - origin_time),
    }


def summarize_stream(stream, trace_type, origin_time, search_window, window_sec=1.0) -> pd.DataFrame:
    rows = [summarize_trace_amplitudes(tr, origin_time, search_window, window_sec) for tr in stream]
    df = pd.DataFrame(rows)
    df["trace_type"] = trace_type
    df = df.sort_values("abs_peak", ascending=False).reset_index(drop=True)

    if trace_type.lower().startswith("infra"):
        # SPL for pressure RMS relative to 20 uPa, using the full search window.
        p_ref = 20e-6
        df["spl_db"] = 20 * np.log10(df["rms"] / p_ref)
        df.loc[~np.isfinite(df["spl_db"]), "spl_db"] = np.nan

    return df


df_infra = summarize_stream(st_infra_bp, "infrasound", EXPLOSION_TIME, INFRASOUND_SEARCH, WINDOW_SEC)
df_seis = summarize_stream(st_seis_bp, "seismic", EXPLOSION_TIME, SEISMIC_SEARCH, WINDOW_SEC)

infra_csv = OUTDIR / "newglenn_peak_infrasound_amplitudes.csv"
seis_csv = OUTDIR / "newglenn_peak_seismic_amplitudes.csv"

df_infra.to_csv(infra_csv, index=False)
df_seis.to_csv(seis_csv, index=False)

display(df_infra.head(20))
display(df_seis.head(20))

print(f"Wrote {infra_csv}")
print(f"Wrote {seis_csv}")

## 7. Load station/source geometry and compute distances

In [ ]:
def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [str(c).strip().lower().replace(" ", "_") for c in df.columns]
    return df


def find_column(df: pd.DataFrame, candidates):
    cols = list(df.columns)
    for cand in candidates:
        cand_norm = cand.lower().replace(" ", "_")
        if cand_norm in cols:
            return cand_norm
    for col in cols:
        for cand in candidates:
            if cand.lower().replace(" ", "_") in col:
                return col
    return None


def load_station_geometry(xls: Path, sheet_name: str) -> pd.DataFrame:
    """
    Load station, latitude, and longitude from a flexible Excel sheet.
    """
    if not xls.exists():
        raise FileNotFoundError(xls)

    df = normalize_columns(pd.read_excel(xls, sheet_name=sheet_name))

    station_col = find_column(df, ["station", "sta", "site", "name"])
    lat_col = find_column(df, ["lat", "latitude"])
    lon_col = find_column(df, ["lon", "long", "longitude"])

    if station_col is None or lat_col is None or lon_col is None:
        raise ValueError(f"Could not identify station/lat/lon columns in {xls}:{sheet_name}. Columns={list(df.columns)}")

    out = df[[station_col, lat_col, lon_col]].copy()
    out = out.rename(columns={station_col: "station", lat_col: "latitude", lon_col: "longitude"})
    out["station"] = out["station"].astype(str).str.strip()
    out["latitude"] = pd.to_numeric(out["latitude"], errors="coerce")
    out["longitude"] = pd.to_numeric(out["longitude"], errors="coerce")
    out = out.dropna(subset=["station", "latitude", "longitude"])
    out = out.drop_duplicates(subset=["station"])
    return out


def load_source_geometry(xls: Path, sheet_name: str, source_name: str):
    """
    Load source lat/lon from a launchpad/source sheet.
    """
    if not xls.exists():
        raise FileNotFoundError(xls)

    df = normalize_columns(pd.read_excel(xls, sheet_name=sheet_name))

    name_col = find_column(df, ["pad", "source", "name", "site"])
    lat_col = find_column(df, ["lat", "latitude"])
    lon_col = find_column(df, ["lon", "long", "longitude"])

    if name_col is None or lat_col is None or lon_col is None:
        raise ValueError(f"Could not identify source/name/lat/lon columns in {xls}:{sheet_name}. Columns={list(df.columns)}")

    m = df[name_col].astype(str).str.upper().str.strip() == str(source_name).upper().strip()
    if not np.any(m):
        available = sorted(df[name_col].dropna().astype(str).unique())
        raise ValueError(f"{source_name!r} not found in {xls}:{sheet_name}. Available names include: {available[:20]}")

    row = df.loc[m].iloc[0]
    return float(row[lat_col]), float(row[lon_col])


def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 6371.0 * c


try:
    station_geometry = load_station_geometry(GEOMETRY_XLS, STATION_SHEET)
    print(f"Loaded station geometry from {GEOMETRY_XLS}:{STATION_SHEET}")
except Exception as e:
    print(f"Could not load station geometry: {e}")
    station_geometry = pd.DataFrame(columns=["station", "latitude", "longitude"])

try:
    SOURCE_LAT, SOURCE_LON = load_source_geometry(GEOMETRY_XLS, SOURCE_SHEET, SOURCE_NAME)
    print(f"Loaded source geometry for {SOURCE_NAME}: lat={SOURCE_LAT:.6f}, lon={SOURCE_LON:.6f}")
except Exception as e:
    print(f"Could not load source geometry: {e}")
    print("Set SOURCE_LAT and SOURCE_LON manually in the Settings cell, then rerun this section.")

if not np.isfinite(SOURCE_LAT) or not np.isfinite(SOURCE_LON):
    raise ValueError("SOURCE_LAT/SOURCE_LON are not finite. Set the source coordinates before computing distances.")

station_geometry["distance_km"] = haversine_km(
    SOURCE_LON, SOURCE_LAT,
    station_geometry["longitude"].values,
    station_geometry["latitude"].values,
)

display(station_geometry.sort_values("distance_km"))

In [ ]:
def attach_geometry(df: pd.DataFrame, station_geometry: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["station"] = out["station"].astype(str).str.strip()
    out = out.merge(station_geometry, on="station", how="left")
    return out


df_infra_geo = attach_geometry(df_infra, station_geometry)
df_seis_geo = attach_geometry(df_seis, station_geometry)

df_infra_geo.to_csv(OUTDIR / "newglenn_peak_infrasound_amplitudes_with_geometry.csv", index=False)
df_seis_geo.to_csv(OUTDIR / "newglenn_peak_seismic_amplitudes_with_geometry.csv", index=False)

display(df_infra_geo[["SEED_id", "station", "channel", "distance_km", "abs_peak", "rms", "spl_db", "time_since_origin_s"]].head(20))
display(df_seis_geo[["SEED_id", "station", "channel", "distance_km", "abs_peak", "rms", "time_since_origin_s"]].head(20))

## 8. Spatial amplitude maps

In [ ]:
def scale_symbol_sizes(values, *, mode="area", max_size=900.0, min_size=30.0, reference=None):
    vals = np.asarray(values, dtype=float)
    vals = np.abs(vals)
    vals[~np.isfinite(vals)] = np.nan

    if reference is None:
        reference = np.nanmax(vals) if np.any(np.isfinite(vals)) else np.nan

    if not np.isfinite(reference) or reference <= 0:
        return np.full_like(vals, min_size, dtype=float)

    ratio = np.clip(vals / reference, 0.0, None)

    if mode == "area":
        sizes = max_size * ratio
    elif mode == "radius":
        sizes = max_size * ratio**2
    else:
        raise ValueError("mode must be 'area' or 'radius'")

    return np.where(np.isfinite(sizes), np.maximum(sizes, min_size), min_size)


def station_level_amplitudes(df: pd.DataFrame, metric="abs_peak") -> pd.DataFrame:
    """
    Reduce channel-level metrics to one station-level value by taking the
    maximum value across channels.
    """
    keep = df.dropna(subset=["distance_km", "latitude", "longitude", metric]).copy()
    if keep.empty:
        return keep

    idx = keep.groupby("station")[metric].idxmax()
    return keep.loc[idx].sort_values("distance_km").reset_index(drop=True)


def plot_spatial_amplitudes(df: pd.DataFrame, metric: str, title: str, outfile: Path):
    plot_df = station_level_amplitudes(df, metric=metric)
    if plot_df.empty:
        print(f"No finite geometry/amplitude values for {title}")
        return None, None

    sizes = scale_symbol_sizes(plot_df[metric].values, max_size=900, min_size=60)

    fig, ax = plt.subplots(figsize=(7, 7))
    ax.scatter(station_geometry["longitude"], station_geometry["latitude"], s=25, marker=".", label="USF stations")
    sc = ax.scatter(plot_df["longitude"], plot_df["latitude"], s=sizes, alpha=0.6, edgecolor="k")
    ax.scatter([SOURCE_LON], [SOURCE_LAT], marker="*", s=250, edgecolor="k", label=SOURCE_NAME)

    if LABEL_POINTS:
        for _, row in plot_df.iterrows():
            ax.annotate(row["station"], (row["longitude"], row["latitude"]), xytext=(4, 4),
                        textcoords="offset points", fontsize=9)

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(outfile, dpi=200)
    plt.show()

    print(f"Wrote {outfile}")
    return fig, ax


plot_spatial_amplitudes(df_infra_geo, "abs_peak", "New Glenn infrasound peak amplitude by station",
                        OUTDIR / "map_newglenn_infrasound_peak_amplitude.png")
plot_spatial_amplitudes(df_seis_geo, "abs_peak", "New Glenn seismic peak amplitude by station",
                        OUTDIR / "map_newglenn_seismic_peak_amplitude.png");

## 9. Amplitude versus distance

In [ ]:
def fit_power_law(distance_km, amplitude):
    """
    Fit A = K r^b in log10 space.
    Returns K, b, and a dataframe-ready predicted function.
    """
    r = np.asarray(distance_km, dtype=float)
    a = np.asarray(amplitude, dtype=float)
    mask = np.isfinite(r) & np.isfinite(a) & (r > 0) & (a > 0)
    if np.sum(mask) < 2:
        return np.nan, np.nan

    x = np.log10(r[mask])
    y = np.log10(a[mask])
    b, logK = np.polyfit(x, y, 1)
    return 10**logK, b


def plot_amplitude_distance(df: pd.DataFrame, metric: str, title: str, ylabel: str, outfile: Path):
    plot_df = df.dropna(subset=["distance_km", metric]).copy()
    plot_df = plot_df[(plot_df["distance_km"] > 0) & (plot_df[metric] > 0)]

    if plot_df.empty:
        print(f"No data for {title}")
        return None, None, None

    K, b = fit_power_law(plot_df["distance_km"], plot_df[metric])

    fig, ax = plt.subplots(figsize=(8, 5))
    for ch, g in plot_df.groupby("channel"):
        ax.scatter(g["distance_km"], g[metric], s=60, label=ch, alpha=0.8)

    if np.isfinite(K):
        rline = np.linspace(plot_df["distance_km"].min()*0.95, plot_df["distance_km"].max()*1.05, 200)
        ax.plot(rline, K * rline**b, linestyle="--", label=f"fit: A = {K:.2g} r$^{{{b:.2f}}}$")

    if LABEL_POINTS:
        for _, row in plot_df.iterrows():
            ax.annotate(row["station"], (row["distance_km"], row[metric]), xytext=(4, 4),
                        textcoords="offset points", fontsize=8)

    ax.set_xlabel(f"Distance from {SOURCE_NAME} (km)")
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.3)
    ax.set_title(title)
    ax.legend(loc="best", fontsize=8)
    fig.tight_layout()
    fig.savefig(outfile, dpi=200)
    plt.show()

    print(f"Wrote {outfile}")
    return fig, ax, (K, b)


_, _, infra_amp_model = plot_amplitude_distance(
    df_infra_geo, "abs_peak",
    "New Glenn infrasound amplitude vs distance",
    "Peak pressure amplitude (Pa)",
    OUTDIR / "newglenn_infrasound_amplitude_vs_distance.png",
)

_, _, seis_amp_model = plot_amplitude_distance(
    df_seis_geo, "abs_peak",
    "New Glenn seismic amplitude vs distance",
    "Peak ground velocity (m/s)",
    OUTDIR / "newglenn_seismic_amplitude_vs_distance.png",
)

print(f"Infrasound amplitude model K,b = {infra_amp_model}")
print(f"Seismic amplitude model K,b = {seis_amp_model}")

## 10. Arrival-time / peak-time versus distance

In [ ]:
def station_level_times(df: pd.DataFrame, metric="abs_peak") -> pd.DataFrame:
    """
    One representative channel per station: the channel with the largest amplitude.
    """
    keep = df.dropna(subset=["distance_km", "time_since_origin_s", metric]).copy()
    keep = keep[(keep["distance_km"] > 0) & np.isfinite(keep["time_since_origin_s"])]
    if keep.empty:
        return keep
    idx = keep.groupby("station")[metric].idxmax()
    return keep.loc[idx].sort_values("distance_km").reset_index(drop=True)


def fit_travel_time(distance_km, time_s):
    """
    Fit t = t_shift + r/v.
    Returns velocity_mps, t_shift_s.
    """
    r_m = np.asarray(distance_km, dtype=float) * 1000.0
    t = np.asarray(time_s, dtype=float)
    mask = np.isfinite(r_m) & np.isfinite(t) & (r_m > 0)
    if np.sum(mask) < 2:
        return np.nan, np.nan

    slope, intercept = np.polyfit(r_m[mask], t[mask], 1)
    velocity = 1.0 / slope if slope != 0 else np.nan
    return velocity, intercept


def plot_time_distance(df: pd.DataFrame, title: str, outfile: Path, expected_velocity=None):
    plot_df = station_level_times(df, metric="abs_peak")
    if plot_df.empty:
        print(f"No time-distance data for {title}")
        return None, None, None

    velocity, t_shift = fit_travel_time(plot_df["distance_km"], plot_df["time_since_origin_s"])

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(plot_df["distance_km"], plot_df["time_since_origin_s"], s=80, alpha=0.85)

    if LABEL_POINTS:
        for _, row in plot_df.iterrows():
            ax.annotate(f"{row['station']} {row['channel']}",
                        (row["distance_km"], row["time_since_origin_s"]),
                        xytext=(4, 4), textcoords="offset points", fontsize=8)

    rline = np.linspace(plot_df["distance_km"].min()*0.95, plot_df["distance_km"].max()*1.05, 200)
    if np.isfinite(velocity):
        ax.plot(rline, t_shift + (rline*1000.0)/velocity,
                linestyle="--", label=f"fit: v={velocity:.0f} m/s, t0 shift={t_shift:.1f} s")

    if expected_velocity is not None and expected_velocity > 0:
        # Draw a reference line through the first observed point.
        anchor = plot_df.iloc[0]
        ref_shift = anchor["time_since_origin_s"] - anchor["distance_km"]*1000.0/expected_velocity
        ax.plot(rline, ref_shift + (rline*1000.0)/expected_velocity,
                linestyle=":", label=f"reference: {expected_velocity:.0f} m/s")

    ax.set_xlabel(f"Distance from {SOURCE_NAME} (km)")
    ax.set_ylabel("Peak-window midpoint after origin time (s)")
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    fig.tight_layout()
    fig.savefig(outfile, dpi=200)
    plt.show()

    print(f"Wrote {outfile}")
    return fig, ax, (velocity, t_shift)


_, _, infra_time_model = plot_time_distance(
    df_infra_geo,
    "New Glenn infrasound peak time vs distance",
    OUTDIR / "newglenn_infrasound_time_vs_distance.png",
    expected_velocity=343.0,
)

_, _, seis_time_model = plot_time_distance(
    df_seis_geo,
    "New Glenn seismic peak time vs distance",
    OUTDIR / "newglenn_seismic_time_vs_distance.png",
    expected_velocity=2500.0,
)

print(f"Infrasound time model velocity,t_shift = {infra_time_model}")
print(f"Seismic time model velocity,t_shift = {seis_time_model}")

## 11. Simple explosion forward model

This is intentionally simple. It treats the explosion as a point source at the known pad/source location and predicts:

\[
t_i = t_0 + t_{\mathrm{shift}} + r_i/v
\]

\[
A_i = K r_i^b
\]

The fitted `b` captures geometric spreading plus attenuation/site effects in a crude empirical way. For a first-order explosion simulation, this is usually more useful than pretending we know the true source spectrum, coupling coefficient, path attenuation, and site response.


In [ ]:
def build_model_comparison(df: pd.DataFrame, time_model, amp_model, metric="abs_peak") -> pd.DataFrame:
    plot_df = station_level_times(df, metric=metric).copy()
    velocity, t_shift = time_model
    K, b = amp_model

    if np.isfinite(velocity):
        plot_df["model_time_since_origin_s"] = t_shift + plot_df["distance_km"] * 1000.0 / velocity
        plot_df["time_residual_s"] = plot_df["time_since_origin_s"] - plot_df["model_time_since_origin_s"]
    else:
        plot_df["model_time_since_origin_s"] = np.nan
        plot_df["time_residual_s"] = np.nan

    if np.isfinite(K) and np.isfinite(b):
        plot_df["model_amplitude"] = K * plot_df["distance_km"]**b
        plot_df["amplitude_ratio_obs_model"] = plot_df[metric] / plot_df["model_amplitude"]
        plot_df["amplitude_residual_log10"] = np.log10(plot_df["amplitude_ratio_obs_model"])
    else:
        plot_df["model_amplitude"] = np.nan
        plot_df["amplitude_ratio_obs_model"] = np.nan
        plot_df["amplitude_residual_log10"] = np.nan

    return plot_df.sort_values("distance_km").reset_index(drop=True)


infra_comparison = build_model_comparison(df_infra_geo, infra_time_model, infra_amp_model, metric="abs_peak")
seis_comparison = build_model_comparison(df_seis_geo, seis_time_model, seis_amp_model, metric="abs_peak")

infra_comparison.to_csv(OUTDIR / "newglenn_infrasound_model_comparison.csv", index=False)
seis_comparison.to_csv(OUTDIR / "newglenn_seismic_model_comparison.csv", index=False)

display(infra_comparison[[
    "SEED_id", "station", "channel", "distance_km",
    "time_since_origin_s", "model_time_since_origin_s", "time_residual_s",
    "abs_peak", "model_amplitude", "amplitude_ratio_obs_model", "amplitude_residual_log10"
]])

display(seis_comparison[[
    "SEED_id", "station", "channel", "distance_km",
    "time_since_origin_s", "model_time_since_origin_s", "time_residual_s",
    "abs_peak", "model_amplitude", "amplitude_ratio_obs_model", "amplitude_residual_log10"
]])

In [ ]:
def plot_model_residuals(comp: pd.DataFrame, title_prefix: str, outfile_prefix: str):
    if comp.empty:
        print(f"No comparison rows for {title_prefix}")
        return

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axhline(0, linestyle="--", linewidth=1)
    ax.scatter(comp["distance_km"], comp["time_residual_s"], s=80)
    for _, row in comp.iterrows():
        ax.annotate(row["station"], (row["distance_km"], row["time_residual_s"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_xlabel(f"Distance from {SOURCE_NAME} (km)")
    ax.set_ylabel("Observed - model peak time (s)")
    ax.set_title(f"{title_prefix}: arrival/peak-time residuals")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    path = OUTDIR / f"{outfile_prefix}_time_residuals.png"
    fig.savefig(path, dpi=200)
    plt.show()
    print(f"Wrote {path}")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.axhline(1, linestyle="--", linewidth=1)
    ax.scatter(comp["distance_km"], comp["amplitude_ratio_obs_model"], s=80)
    for _, row in comp.iterrows():
        ax.annotate(row["station"], (row["distance_km"], row["amplitude_ratio_obs_model"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=8)
    ax.set_xlabel(f"Distance from {SOURCE_NAME} (km)")
    ax.set_ylabel("Observed / model amplitude")
    ax.set_yscale("log")
    ax.set_title(f"{title_prefix}: amplitude residuals")
    ax.grid(True, which="both", alpha=0.3)
    fig.tight_layout()
    path = OUTDIR / f"{outfile_prefix}_amplitude_residuals.png"
    fig.savefig(path, dpi=200)
    plt.show()
    print(f"Wrote {path}")


plot_model_residuals(infra_comparison, "New Glenn infrasound", "newglenn_infrasound")
plot_model_residuals(seis_comparison, "New Glenn seismic", "newglenn_seismic")

## 12. Optional: estimate a more explicit seismic velocity from manual picks

In [ ]:
# The peak-window midpoint is useful, but it is not the same as a first arrival.
# For seismic velocity, manual first-break picks are better.
#
# Create a CSV like:
# station,channel,pick_time_utc
# B05,DHZ,2026-05-29T01:00:37.123
# B29,DHZ,2026-05-29T01:00:42.456
#
# Then set PICK_CSV to that file and run this cell.

PICK_CSV = OUTDIR / "newglenn_manual_seismic_picks.csv"

if PICK_CSV.exists():
    picks = pd.read_csv(PICK_CSV)
    picks["station"] = picks["station"].astype(str).str.strip()
    picks = picks.merge(station_geometry, on="station", how="left")

    pick_times = [UTCDateTime(t) for t in picks["pick_time_utc"]]
    picks["pick_time_since_origin_s"] = [float(t - EXPLOSION_TIME) for t in pick_times]

    v_pick, t_shift_pick = fit_travel_time(picks["distance_km"], picks["pick_time_since_origin_s"])
    print(f"Manual-pick seismic velocity = {v_pick:.0f} m/s; t0 shift = {t_shift_pick:.2f} s")

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(picks["distance_km"], picks["pick_time_since_origin_s"], s=80)
    for _, row in picks.iterrows():
        ax.annotate(row["station"], (row["distance_km"], row["pick_time_since_origin_s"]),
                    xytext=(4, 4), textcoords="offset points", fontsize=8)

    rline = np.linspace(picks["distance_km"].min()*0.95, picks["distance_km"].max()*1.05, 200)
    ax.plot(rline, t_shift_pick + rline*1000/v_pick, linestyle="--",
            label=f"fit: v={v_pick:.0f} m/s")
    ax.set_xlabel(f"Distance from {SOURCE_NAME} (km)")
    ax.set_ylabel("Manual pick after origin time (s)")
    ax.set_title("New Glenn manual seismic first-break picks")
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTDIR / "newglenn_manual_seismic_picks_velocity_fit.png", dpi=200)
    plt.show()

    display(picks)
else:
    print(f"No manual pick CSV found yet: {PICK_CSV}")
    print("Create this file after visual picking if you want a better seismic velocity estimate.")